### Import Libaries

In [ ]:
import passwords as pw
import flickrapi
import pandas as pd
import pprint
import folium

api_key = pw.flickr_key
api_secret = pw.flickr_secret

# connect to flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format="parsed-json")

### Set search parameters

In [ ]:
# Define the location (latitude and longitude) and search parameters
centre_latitude = 45.436814
centre_longitude = 12.336943

# Define the bounding box. Use Open Street Map to get the coordinates
lat_north = 45.44996
lat_south = 45.42267
long_west = 12.30031
long_east = 12.36958

circle_radius = 2

# Format: bbox = "min_longitude,min_latitude,max_longitude,max_latitude" for Flickr search
bbox = str(long_west)+','+str(lat_south)+','+str(long_east)+','+str(lat_north)

# Define the number of photos to retrieve per page
per_page = 250

# Create a map centered on the location
map_Venice = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=14,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False,
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=circle_radius,
    color="HotPink",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="{} pixels".format(circle_radius), # <--- CHANGE 'radius' to 'circle_radius'
).add_to(map_Venice)

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north, long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="HotPink",
).add_to(map_Venice)

map_Venice

### Get amount of pages and photos

In [ ]:
photos = flickr.photos.search(bbox=bbox, per_page=per_page, page=1, has_geo=1)

total_pages = photos["photos"]["pages"]
total_photos = photos["photos"]["total"]

print(f"Total photos: {total_photos}")
print(f"Total pages: {total_pages}")

### Extract information

In [ ]:
def get_page(bbox, page, per_page):
    response = flickr.photos.search(
        bbox=bbox,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,views,media,url_s,date_taken,owner_name",
    )
    photos = response["photos"]["photo"]
    rows = []
    for photo in photos:
        new_row = {
            "id": photo["id"],
            "server": photo["server"],
            "secret": photo["secret"],
            "title": photo["title"],
            "tags": photo["tags"],
            "views": photo["views"],
            "description": photo["description"]["_content"],
            "date_taken": photo["datetaken"],
            "latitude": photo["latitude"],
            "longitude": photo["longitude"],
            "url_s": photo["url_s"],
            "owner": photo["owner"],
            "owner_name": photo["ownername"],
            "media": photo["media"],
        }
        rows.append(new_row)
    df = pd.DataFrame(
        rows,
        columns=[
            "id",
            "server",
            "secret",
            "title",
            "tags",
            "views",
            "description",
            "date_taken",
            "latitude",
            "longitude",
            "url_s",
            "owner_name",
            "owner",
            "media",
        ],
    )
    return df


df = pd.DataFrame()

# Use this code to get only the first 10 pages of photos
for page in range(1, 1124):
    new_df = get_page(bbox, page, per_page)
    print(f"Getting page {page} of {total_pages}")
    df = pd.concat([df, new_df], ignore_index=True)
    print(f"Total photos so far: {len(df)}")

""" 
Use this code to get all the photos in the bounding box: 
for page in range(1, total_pages + 1):
    new_df = get_page(bbox, page, per_page)
    print(f"Getting page {page} of {total_pages}")
    df = pd.concat([df, new_df], ignore_index=True)
    print(f"Total photos so far: {len(df)}") 
"""

df.to_csv("Venice_photos.csv", index=False)

df

#Visualizing data on folium map

In [ ]:
import pandas as pd
import folium

# --- 1. Define Location and Bounding Box (Based on Venice) ---

# Center coordinates for the map (approximately the center of Venice)
centre_latitude = 45.434827
centre_longitude = 12.348785

# Bounding Box Coordinates (defining the area of interest)
# These should roughly encompass the area used during scraping
lat_north = 45.44996
lat_south = 45.42267
long_west = 12.30031
long_east = 12.36958

# Define the radius for the central marker (to avoid NameError)
main_marker_radius = 2


# --- 2. Load Data from CSV ---

try:
    # Load the data from the CSV file you exported
    # Ensure this file exists in the same directory as this script
    df = pd.read_csv("Venice_photos.csv")

    # Filter out rows where latitude or longitude might be missing (just in case)
    df = df.dropna(subset=['latitude', 'longitude'])

    print(f"Successfully loaded {len(df)} photos from the CSV file.")

except FileNotFoundError:
    print("ERROR: 'Venice_photos.csv' not found.")
    print("Please make sure the file is in the correct directory.")
    # Exit gracefully if the file isn't found
    df = pd.DataFrame({'latitude': [], 'longitude': []})
    map_Venice = folium.Map(location=[centre_latitude, centre_longitude], zoom_start=14)
    map_Venice.save("venice_map_error.html")
    # We can't proceed with plotting, so we stop here.
    exit()

# --- 3. Create the Base Folium Map ---

map_Venice = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter", # A nice dark theme for visualization
    zoom_start=14,
    control_scale=True,
    zoom_control=True, # Keeping zoom control on for easier navigation
    dragging=True,     # Allowing dragging
    scrollWheelZoom=True, # Allowing scroll wheel zoom
)

# --- 4. Add Decorative Markers (Center and Bounding Box) ---

# Add a circle marker at the center location
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=main_marker_radius,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="Center Point ({} pixels)".format(main_marker_radius),
).add_to(map_Venice)

# Add a rectangle representing the bounding box
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",
).add_to(map_Venice)


# --- 5. Add a Circle Marker for Each Photo ---

# Extract latitude and longitude columns
latitudes = df['latitude']
longitudes = df['longitude']

# Iterate over all photo coordinates and add markers
for latitude, longitude in zip(latitudes, longitudes):
    coordinate = [latitude, longitude]
    
    # We use a small radius for the individual photo markers
    photo_radius = 1 
    
    # You could also use a different marker type, like folium.Marker, 
    # but CircleMarker is great for dense scatter plots
    folium.CircleMarker(
        location=coordinate,
        radius=photo_radius,
        stroke=False,
        fill=True,
        fillColor="orchid",
        fill_opacity=0.5,
        opacity=0.7,
    ).add_to(map_Venice)

# --- 6. Display the Map (or save it) ---

# If you are running this in a Jupyter/Colab environment, the map will display automatically.
# Otherwise, you can save it as an interactive HTML file:
map_Venice.save("venice_photos_map.html")

print("\nMap generation complete. Look for 'venice_photos_map.html' in your directory.")
print("The interactive map object is stored in the 'map_Venice' variable.")

# Optional: Return the map object for notebook display
map_Venice

In [ ]:
#Generating heatmap

In [ ]:
import pandas as pd
import folium
# Import the plugins module for the HeatMap
from folium.plugins import HeatMap 
# Import for high-resolution export (install separately: pip install selenium pillow)
from selenium import webdriver 
from PIL import Image

# --- 1. Define Location and Bounding Box (Based on Venice) ---
# ... (rest of the coordinates and variable definitions are the same)
centre_latitude = 45.434827
centre_longitude = 12.348785

lat_north = 45.44996
lat_south = 45.42267
long_west = 12.30031
long_east = 12.36958

main_marker_radius = 2


# --- 2. Load Data from CSV ---
# ... (rest of the file loading and error handling is the same)
try:
    df = pd.read_csv("Venice_photos.csv")
    df = df.dropna(subset=['latitude', 'longitude'])
    print(f"Successfully loaded {len(df)} photos from the CSV file.")

except FileNotFoundError:
    print("ERROR: 'Venice_photos.csv' not found. Exiting.")
    exit()

# --- 3. Create the Base Folium Map ---
# ... (map creation is the same)
map_Venice = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter", 
    zoom_start=14,
    control_scale=True,
    zoom_control=True, 
    dragging=True, 
    scrollWheelZoom=True,
)

# --- 4. Add Decorative Markers (Center and Bounding Box) ---
# ... (decorative markers are the same)
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=main_marker_radius,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="Center Point ({} pixels)".format(main_marker_radius),
).add_to(map_Venice)

folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",
).add_to(map_Venice)


# --- 5. Add a HeatMap for Density (Replacing Individual Markers) ---

# Prepare the data in the required format: list of [latitude, longitude]
# You can optionally add a weight (density) column, but default (1) is usually fine.
heat_data = df[['latitude', 'longitude']].values.tolist()

HeatMap(
    heat_data,
    name='Photo Density HeatMap',
    min_opacity=0.2, # Start showing density at lower values
    max_val=1.0,     # Max value in the scale
    radius=12,       # Radius of influence for each point (adjust for 'dot' size)
    blur=10,         # Smoothness of the gradient (adjust for more distinct dots)
    gradient={0.4: 'blue', 0.65: 'lime', 1.0: 'red'}, # Custom gradient for color contrast
).add_to(map_Venice)


# --- 6. Save the Map HTML ---

html_output_file = "venice_photos_heatmap.html"
map_Venice.save(html_output_file)
print(f"\nHeatMap HTML generation complete. Look for '{html_output_file}' in your directory.")

# --- 7. Export to High-Resolution PNG using Selenium ---

def save_folium_map_as_png(html_path, png_path, resolution=(1920, 1080)):
    """Renders the HTML map using a browser and saves it as a PNG."""
    print(f"Attempting to capture map as PNG: {png_path}...")
    
    # You will need a browser driver (e.g., ChromeDriver, GeckoDriver) 
    # and to specify its path, or ensure it's in your PATH.
    # We use Chrome here as an example.
    try:
        # Set up Chrome options for headless mode and desired window size
        options = webdriver.ChromeOptions()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument(f'--window-size={resolution[0]},{resolution[1]}')

        # Initialize the driver (ensure you have the correct driver installed)
        driver = webdriver.Chrome(options=options)
        
        # Open the local HTML file
        driver.get(f'file:///{os.path.abspath(html_path)}')
        
        # Give the map time to fully load all tiles and layers (essential for Folium)
        import time
        time.sleep(3) 

        # Take the screenshot
        driver.save_screenshot(png_path)
        print(f"Successfully exported high-resolution PNG: {png_path}")

    except Exception as e:
        print(f"ERROR: Failed to generate PNG. This usually means Selenium/WebDriver is not set up correctly.")
        print("Please ensure you have:")
        print("1. Installed the necessary packages: 'pip install folium pandas selenium pillow'")
        print("2. Downloaded a WebDriver (e.g., ChromeDriver) and it's accessible in your system's PATH.")
        print(f"Details: {e}")
    finally:
        if 'driver' in locals() and driver:
            driver.quit()

import os
save_folium_map_as_png(html_output_file, "venice_photos_heatmap.png", resolution=(1920, 1080))

print("\nProcess finished.")

In [ ]:
#Generating grids of dots

In [ ]:
import pandas as pd
import folium

# --- 1. Define Location and Bounding Box (Based on Venice) ---

# Center coordinates for the map (approximately the center of Venice)
centre_latitude = 45.434827
centre_longitude = 12.348785

# Bounding Box Coordinates (defining the area of interest)
# These should roughly encompass the area used during scraping
lat_north = 45.44996
lat_south = 45.42267
long_west = 12.30031
long_east = 12.36958

# Define the radius for the central marker (to avoid NameError)
main_marker_radius = 2


# --- 2. Load Data from CSV ---

try:
    # Load the data from the CSV file you exported
    # Ensure this file exists in the same directory as this script
    df = pd.read_csv("Venice_photos.csv")

    # Filter out rows where latitude or longitude might be missing (just in case)
    df = df.dropna(subset=['latitude', 'longitude'])

    print(f"Successfully loaded {len(df)} photos from the CSV file.")

except FileNotFoundError:
    print("ERROR: 'Venice_photos.csv' not found.")
    print("Please make sure the file is in the correct directory.")
    # If the file is not found, we create a basic map and stop execution.
    df = pd.DataFrame({'latitude': [], 'longitude': []})
    map_Venice = folium.Map(location=[centre_latitude, centre_longitude], zoom_start=14)
    map_Venice.save("venice_map_error.html")
    exit() # Stop execution here as data is required

# --- 3. Create the Base Folium Map ---

map_Venice = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter", # A nice dark theme for visualization
    zoom_start=14,
    control_scale=True,
    zoom_control=True, 
    dragging=True,     
    scrollWheelZoom=True,
)

# --- 4. Add Decorative Markers (Center and Bounding Box) ---

# Add a circle marker at the center location
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=main_marker_radius,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="Center Point ({} pixels)".format(main_marker_radius),
).add_to(map_Venice)

# Add a rectangle representing the bounding box
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",
).add_to(map_Venice)


# --- 5. Generate and Plot Density Grid (Dot Matrix) ---

# Define grid size: 0.001 degrees is roughly 100 meters, creating a distinct grid cell
GRID_RESOLUTION = 0.001 

# Bin the data by rounding coordinates to create the grid cells (lat_bin, lon_bin)
df['lat_bin'] = (df['latitude'] / GRID_RESOLUTION).round() * GRID_RESOLUTION
df['lon_bin'] = (df['longitude'] / GRID_RESOLUTION).round() * GRID_RESOLUTION

# Calculate the density (count of photos) for each grid cell
density_df = df.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='count')

# Calculate the maximum count to normalize the color and size scales
max_count = density_df['count'].max()

# Define the color scale function (light yellow -> dark red)
def get_color(count, max_count):
    """Maps photo count to a specific color intensity."""
    normalized = count / max_count
    
    if normalized < 0.1:
        return '#FFFFB3' # Very Low: Light Yellow
    elif normalized < 0.3:
        return '#FECC5C' # Low: Light Orange
    elif normalized < 0.5:
        return '#FD8D3C' # Medium: Orange
    elif normalized < 0.7:
        return '#E34A33' # High: Red
    else:
        return '#B30000' # Very High: Dark Red

# Iterate through the density data and plot colored circle markers for each bin
for index, row in density_df.iterrows():
    # Use the center of the bin for plotting
    bin_center = [row['lat_bin'], row['lon_bin']]
    count = row['count']
    
    folium.CircleMarker(
        location=bin_center,
        # Scale the radius based on the count (max radius is 8)
        radius=2 + (count / max_count) * 6, 
        stroke=False,
        fill=True,
        fillColor=get_color(count, max_count),
        fill_opacity=0.8,
        opacity=0.8,
        popup=f"Photos: {count}",
    ).add_to(map_Venice)

# --- 6. Display the Map (or save it) ---

# The map will display automatically in a notebook environment.
map_Venice.save("venice_photos_map2.html")

print("\nMap generation complete. Look for 'venice_photos_map.html' in your directory.")
print("The map now uses a dot matrix approach to visualize photo intensity.")